# OSC AlphaGenome setup
## imports

In [47]:
from alphagenome_research.model import dna_model
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript
from alphagenome.data import ontology
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from pysam import VariantFile
import pysam
from io import StringIO
from tqdm import tqdm
import os
import gc
from cyvcf2 import VCF, Writer
import gffpandas.gffpandas as gffpd
import pybedtools

# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='true'
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9' # pre allocates XX% of total GPU instead of the default 75%

pd.set_option('display.max_columns', None)

## common variables

In [5]:
LMNA_START = 156_082_572
LMNA_END = 156_140_081
gene_symbol = "LMNA"
LMNA_INTERVAL = genome.Interval('chr1', 156_082_572, 156_140_081)


BASE_PATH = '/users/PAS2905/coraalbers/'
AG_DATA_PATH = '/users/PAS2905/coraalbers/ag/ag_data/'

HG38_FASTA_PATH = '/users/PAS2905/coraalbers/ag/hg38.fa'
HG38_GTF_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf.gz.feather'
HG38_SPLICE_START_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_starts.feather'
HG38_SPLICE_END_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_ends.feather'

CLINVAR_PATH = '/users/PAS2905/coraalbers/ag/clinvar.vcf.gz'

gtf = pd.read_feather( 'https://storage.googleapis.com/alphagenome/reference/gencode/' 'hg38/gencode.v46.annotation.gtf.gz.feather' )

output_modalities = ['atac',	
    'cage',	
    'chip_histone',	
    'chip_tf',	
    'contact_maps',	
    'dnase',	
    'procap',	
    'rna_seq',	
    'splice_junctions',	
    'splice_site_usage',	
    'splice_sites']

requested_outputs = {dna_client.OutputType.ATAC,
        dna_client.OutputType.CAGE,
        dna_client.OutputType.DNASE,
        dna_client.OutputType.PROCAP,
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.SPLICE_SITES,
        dna_client.OutputType.SPLICE_SITE_USAGE,
        dna_client.OutputType.SPLICE_JUNCTIONS,
        dna_client.OutputType.CONTACT_MAPS,
        dna_client.OutputType.CHIP_HISTONE,
        dna_client.OutputType.CHIP_TF}

## model initialization

In [3]:
model = dna_model.create_from_huggingface( 
    'all_folds', 
    organism_settings={ 
        dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings( 
            fasta_path=HG38_FASTA_PATH, 
            gtf_feather_path=HG38_GTF_PATH, 
            splice_site_starts_feather_path=HG38_SPLICE_START_PATH, 
            splice_site_ends_feather_path=HG38_SPLICE_END_PATH, 
        ), dna_model.Organism.MUS_MUSCULUS: dna_model.OrganismSettings() } )

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

# filter ClinVar database
## filter to only LMNA affecting variants within 500 kb of gene body

In [33]:
PATHOGENIC = {
    "Pathogenic",
    "Likely_pathogenic",
    "Pathogenic/Likely_pathogenic"    
}

BENIGN = {
    'Benign',
    'Likely_benign',
    'Benign/Likely_benign'
}

# define variant range (500 kb up and downstream from gene)
window_bp = 500000
vcf_range_start = LMNA_START - window_bp
vcf_range_end = LMNA_END + window_bp
vcf_range = f'1:{vcf_range_start}-{vcf_range_end}'


vcf = VCF(CLINVAR_PATH)

#### change variables here!!!
clinical_significance = BENIGN
output_vcf = "outputs/clinvar_LMNA.BLB.vcf"


In [34]:
# create a new vcf Writer using the input vcf as a template.
w = Writer(output_vcf, vcf)


for v in vcf(vcf_range):
    # print(v)
    clnsig = v.INFO.get("CLNSIG")
    # print(clnsig)
    if 'LMNA:' in v.INFO["GENEINFO"] and clnsig is not None:
        if any(item in v.INFO["CLNSIG"] for item in clinical_significance):
            w.write_record(v)


## filter to only noncoding variants
**get non-coding region coords using gtftools (get_noncoding_coords notebook)**

In [49]:
# annotation = gffpd.read_gff3(f'{AG_DATA_PATH}gencode.v46.annotation.gtf')
annotation.stats_dic()

/users/PAS2905/coraalbers/.conda/envs/py311/lib/python3.11/site-packages/gffpandas/gffpandas.py:180: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  strand_counts = pd.value_counts(self.df['strand']).to_dict()
/users/PAS2905/coraalbers/.conda/envs/py311/lib/python3.11/site-packages/gffpandas/gffpandas.py:181: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  type_counts = pd.value_counts(self.df['type']).to_dict()


{'Maximal_bp_length': np.int64(2473538),
 'Minimal_bp_length': np.int64(0),
 'Counted_strands': {'+': 1763768, '-': 1703388},
 'Counted_feature_types': {'exon': 1668627,
  'CDS': 899146,
  'UTR': 390193,
  'transcript': 254070,
  'start_codon': 98995,
  'stop_codon': 92909,
  'gene': 63086,
  'Selenocysteine': 130}}

In [51]:
plp_vcf = f'{BASE_PATH}ag/variant-effects/osc/outputs/clinvar_LMNA.PLP.vcf'
blb_vcf = f'{BASE_PATH}ag/variant-effects/osc/outputs/clinvar_LMNA.BLB.vcf'

noncoding = pybedtools.BedTool(f'{AG_DATA_PATH}noncoding.bed')
plp = pybedtools.BedTool(plp_vcf)


In [58]:
noncoding

<BedTool(/users/PAS2905/coraalbers/ag/ag_data/noncoding.bed)>

In [59]:

# bedtools intersect -wo -a /users/PAS2905/coraalbers/ag/variant-effects/osc/outputs/clinvar_LMNA.PLP.vcf -b /users/PAS2905/coraalbers/ag/ag_data/noncoding_num_chr.bed > outputs/plp_with_nc.bed
# bedtools intersect -wo -a /users/PAS2905/coraalbers/ag/variant-effects/osc/outputs/clinvar_LMNA.BLB.vcf -b /users/PAS2905/coraalbers/ag/ag_data/noncoding_num_chr.bed > outputs/blb_with_nc.bed

In [ ]:
## run from ag/ directory, only needed to be run once
# gatk CreateSequenceDictionary -R hg38.fa